# Chapter 7 — CNNs · *PyTorch Companion*
**Track:** ML from Scratch · Synthetic neighbourhood grid images

> This is the PyTorch companion to [notebook.ipynb](notebook.ipynb). NumPy sections are
> preserved verbatim. Every TensorFlow/Keras cell is followed by a runnable **PyTorch
> equivalent** using `nn.Module` and an explicit training loop.
>
> Layout convention: **Keras cell → "PyTorch equivalent" markdown → PyTorch cell**.
> Hyperparameters, seeds, and batch sizes match the Keras version so results are comparable.

## Core Idea

A **Convolutional Neural Network** applies learned filters that slide across the input — sharing the same weights at every spatial position.

```
Dense (Ch.4): 64 × 128 + 128 = 8,320 params for first layer
CNN: (3×3×1 + 1) × 8 = 80 params for equivalent first conv layer
```

Key ingredients: **Convolution → ReLU → Pooling**, repeated, then **Dense head**.

## Running Example

Synthetic **8×8 neighbourhood grid images** (greyscale):
- `label=0` (tidy): high mean brightness, low variance
- `label=1` (distressed): low mean brightness, high variance (patchy lots)

This mirrors real property-condition classification from aerial imagery — same task, fully self-contained without downloading external data.

In [ ]:
# TODO: Implement this cell
# Hint:
#   import numpy as np
#   import matplotlib.pyplot as plt
#   from sklearn.model_selection import train_test_split
#   from sklearn.metrics import accuracy_score, classification_report


In [ ]:
# TODO: Implement this cell
# Hint:
#   fig, axes = plt.subplots(2, 8, figsize=(14, 3.5))
#   for cls, label in [(0, 'Tidy'), (1, 'Distressed')]:
#       samples = X_img[y_img == cls][:8]
#       for j, ax in enumerate(axes[cls]):


## Manual 2D Convolution (NumPy)

$$
(\mathbf{X} * \mathbf{K})_{i,j} = \sum_{u=0}^{k-1}\sum_{v=0}^{k-1} \mathbf{X}_{i+u,\,j+v} \cdot \mathbf{K}_{u,v}
$$

Output size (no padding, stride 1): $H_\text{out} = H - k + 1$

In [ ]:
# TODO: Implement this cell
# Hint:
#   def conv2d_numpy(x, kernel, stride=1, padding=0):
#       """Manual 2D cross-correlation (single channel).
#       x:      (H, W)
#       kernel: (k, k)


## Filter Visualisation — What Different Kernels Detect

Different hand-crafted kernels detect different patterns. CNNs learn these kernels automatically from data.

In [ ]:
# TODO: Implement this cell
# Hint:
#   kernels = {
#       'Horizontal edges': np.array([[-1,-1,-1],[0,0,0],[1,1,1]], dtype=np.float...
#       'Vertical edges':   np.array([[-1,0,1],[-1,0,1],[-1,0,1]], dtype=np.float...
#       'Sharpen':          np.array([[0,-1,0],[-1,5,-1],[0,-1,0]], dtype=np.floa...


## Parameter Count: Dense vs CNN

Why do CNNs need far fewer parameters for the same input?

In [ ]:
# TODO: Implement this cell
# Hint:
#   input_pixels = 1 * 8 * 8
#   dense_params = input_pixels * 128 + 128
#   conv_params = (3 * 3 * 1 + 1) * 8
#   print(f"Dense (64 → 128):           {dense_params:,} parameters")


## CNN Classifier with Keras

Build and train a CNN on the synthetic neighbourhood grids.
Architecture: `Input(1,8,8) → Conv(8,3×3) → MaxPool → Conv(16,3×3) → Flatten → Dense(32) → Dense(1, sigmoid)`

In [ ]:
# TODO: Implement this cell
# Hint:
#   try:
#       import tensorflow as tf
#       from tensorflow import keras
#       from tensorflow.keras import layers


### PyTorch equivalent — setup

| Keras | PyTorch |
|---|---|
| `import tensorflow as tf; from tensorflow import keras` | `import torch; import torch.nn as nn; import torch.nn.functional as F` |
| `tf.random.set_seed(42)` | `torch.manual_seed(42)` |
| channels-**last** `(N, H, W, C)` | channels-**first** `(N, C, H, W)` — **no transpose needed**, our NumPy arrays already match |
| automatic device placement | explicit `.to(device)` |

In [ ]:
# TODO: Implement this cell
# Hint:
#   try:
#       import torch
#       import torch.nn as nn
#       import torch.nn.functional as F


In [ ]:
# TODO: Implement this cell
# Hint:
#   if TF_AVAILABLE:
#       tf.random.set_seed(42)
#       X_tr_cl = X_tr.transpose(0, 2, 3, 1)   # (N, 1, 8, 8) → (N, 8, 8, 1)
#       X_te_cl = X_te.transpose(0, 2, 3, 1)


### PyTorch equivalent — CNN model

Keras `Sequential` → PyTorch `nn.Module` subclass. The explicit `forward` method makes shape
transitions visible, which is the main pedagogical win over `Sequential` for a CNN.

| Keras | PyTorch |
|---|---|
| `layers.Conv2D(8, 3, padding='valid')` | `nn.Conv2d(in_channels=1, out_channels=8, kernel_size=3)` — padding defaults to 0 (= `'valid'`) |
| `layers.MaxPooling2D(2)` | `nn.MaxPool2d(2)` |
| `layers.Flatten()` | `x.flatten(1)` in `forward` (keep batch dim) |
| `layers.Dense(n, activation='relu')` | `nn.Linear(in, n)` + `F.relu(...)` |
| `activation='sigmoid'` on last layer | raw logits + `nn.BCEWithLogitsLoss()` (numerically safer than `sigmoid` + BCE) |
| `cnn.summary()` | manual param count via `sum(p.numel() for p in model.parameters())` |

In [ ]:
# TODO: Implement this cell
# Hint:
#   if TORCH_AVAILABLE:
#       class PropertyConditionCNN(nn.Module):
#           """Same architecture as the Keras model above, in PyTorch.
#           Input:  (N, 1, 8, 8)  → Conv(1→8,3×3)   → (N, 8, 6, 6)


In [ ]:
# TODO: Implement this cell
# Hint:
#   if TF_AVAILABLE:
#       history = cnn.fit(
#           X_tr_cl, y_tr,
#           epochs=40,


### PyTorch equivalent — training loop

`model.fit(...)` is the big "magic" to replace. A PyTorch loop has 5 explicit steps per batch:
`zero_grad → forward → loss → backward → optimizer.step`. This verbosity is the *point* — it
makes custom tricks (mixed precision, gradient clipping, manual scheduler, multi-task losses)
trivial to add without fighting the framework.

Same hyperparameters as Keras: `Adam`, `lr=1e-3`, 40 epochs, batch size 64, 15% validation split.

In [ ]:
# TODO: Implement this cell
# Hint:
#   if TORCH_AVAILABLE:
#       n_val   = int(0.15 * len(X_tr))
#       X_tr_t  = torch.tensor(X_tr[n_val:], dtype=torch.float32, device=device) ...
#       y_tr_t  = torch.tensor(y_tr[n_val:], dtype=torch.float32, device=device)


## Learned Filter Visualisation

After training, the first conv layer's 8 filters have been learned from data. Visualise them and their responses.

In [ ]:
# TODO: Implement this cell
# Hint:
#   if TF_AVAILABLE:
#       conv1_weights = cnn.layers[0].get_weights()[0]   # (3,3,1,8)
#       fig, axes = plt.subplots(2, 8, figsize=(14, 4))
#       for f in range(8):


### PyTorch equivalent — filter visualisation

Keras stores conv kernels as `(k_h, k_w, in_channels, out_channels)`. PyTorch stores them as
`(out_channels, in_channels, k_h, k_w)`. Index accordingly when visualising.

In [ ]:
# TODO: Implement this cell
# Hint:
#   if TORCH_AVAILABLE:
#       conv1_w_torch = cnn_torch.conv1.weight.detach().cpu().numpy()
#       fig, axes = plt.subplots(2, 8, figsize=(14, 4))
#       for f in range(8):


## Filter Depth Sweep — How Many Filters?

Vary the number of filters in the first conv layer and track test accuracy.

In [ ]:
# TODO: Implement this cell
# Hint:
#   if TF_AVAILABLE:
#       filter_counts = [2, 4, 8, 16, 32]
#       filter_accs = []
#       for n_filters in filter_counts:


### PyTorch equivalent — filter depth sweep

In [ ]:
# TODO: Implement this cell
# Hint:
#   if TORCH_AVAILABLE:
#       def build_cnn_nfilters(n_filters):
#           """Tiny CNN matching the Keras sweep model: Conv(1→n)→Pool→Flatten→De...
#           return nn.Sequential(


## Trap 1 — Kernel Size Too Large for Small Inputs

A 5×5 kernel on an 8×8 input (no padding, stride 1) produces a 4×4 output. After MaxPool(2×2) → 2×2. One more conv(3×3): output is 0×0 — the network crashes.

In [ ]:
# TODO: Implement this cell
# Hint:
#   import math
#   def output_size(h, k, s=1, p=0):
#       return math.floor((h + 2*p - k) / s) + 1
#   print(f"{'Kernel':>8} {'After Conv1':>12} {'After Pool':>12} {'After Conv2':>...


## Trap 2 — Unscaled Pixel Values

If pixel values are in [0, 255] instead of [0, 1], the first conv layer receives activations ~100× larger than expected. Gradients from the first layer are correspondingly larger → unstable training.

In [ ]:
# TODO: Implement this cell
# Hint:
#   X_unscaled = X_img * 255.0
#   rng = np.random.default_rng(42)
#   rand_filter = rng.normal(0, 1, (3, 3)).astype(np.float32)
#   sample_scaled   = X_img[0, 0]         # [0, 1]


## Exercises

**Exercise 1 — Global Average Pooling vs Flatten**
Replace the `Flatten()` layer with `GlobalAveragePooling2D()` in the Keras model. Compare parameter counts and test accuracy. When would GAP be preferable?

**Exercise 2 — Receptive field calculator**
Implement a function `receptive_field(n_layers, k)` that computes $1 + n \times (k-1)$. Plot the receptive field for `k=3` and `n_layers` from 1 to 10. At what depth does the receptive field exceed the 8×8 input?

**Exercise 3 — Custom edge detector**
Generate a batch of synthetic grids where the tidy class has strong horizontal banding and the distressed class has random noise. Design a hand-crafted horizontal-edge kernel that achieves > 80% accuracy using only threshold classification on the mean filter response (no neural network needed). What does this tell you about the role of prior knowledge vs learned features?

In [ ]:
# TODO: Implement this cell
# Hint:
#   if TF_AVAILABLE:
#       cnn_gap = keras.Sequential([
#           layers.Input(shape=(8, 8, 1)),
#           layers.Conv2D(8, 3, activation='relu', padding='valid'),


### PyTorch equivalent — Exercise 1 (Global Average Pooling)

PyTorch has no single `GlobalAveragePooling2D` layer — use `nn.AdaptiveAvgPool2d(1)` (which
reduces any spatial size to 1×1) followed by `flatten(1)`.

In [ ]:
# TODO: Implement this cell
# Hint:
#   if TORCH_AVAILABLE:
#       class CNNWithGAP(nn.Module):
#           def __init__(self):
#               super().__init__()


In [ ]:
# TODO: Implement this cell
# Hint:
#   def receptive_field(n_layers, k, stride=1):
#       """Receptive field after n_layers of conv with kernel k and given stride."""
#       return 1 + n_layers * (k - 1)  # stride=1, padding=0 formula
#   layers_range = range(1, 11)


In [ ]:
# TODO: Implement this cell
# Hint:
#   def make_banded_grids(n_samples=500, grid_size=8, seed=42):
#       """Tidy = horizontal bands; distressed = random noise."""
#       rng = np.random.default_rng(seed)
#       X, y = [], []
